# تحلیل رفتار حقیقی و حقوقی بورس ایران

این دفترچه ابزار شخصی و محلی تحلیل داده‌های بازار است. کدها با Python و Pandas اجرا می‌شوند، نمودارها با Plotly داخل همین Notebook نمایش داده می‌شوند و داده‌ها در فایل Excel محلی نگهداری می‌شوند.

**هیچ سرور یا داشبورد وبی اجرا نمی‌شود.** دریافت اطلاعات از TSETMC فقط با اجرای سلول همگام‌سازی و در صورت دسترسی شبکه انجام می‌شود.

## ۱. آماده‌سازی و بارگذاری ابزارها

این سلول را ابتدا اجرا کن. سپس سلول‌های دیگر را به‌ترتیب و فقط در زمان نیاز اجرا کن.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "smart_money").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display

from smart_money.analytics import (
    HORIZONS,
    backtest_patterns,
    build_signal_history,
    detect_latest_patterns,
    enrich_market_data,
)
from smart_money.storage import (
    WORKBOOK_PATH,
    add_instrument,
    load_book,
    save_book,
)
from smart_money.sync import sync_all
from smart_money.tsetmc import search_instruments

# Plotly uses Jupyter's MIME renderer; it does not need an external CDN.
pio.renderers.default = "plotly_mimetype"

print(f"مسیر پروژه: {PROJECT_ROOT}")
print(f"فایل داده: {WORKBOOK_PATH}")

## ۲. تنظیمات و فایل داده

فایل Excel در اولین ذخیره ساخته می‌شود. اجرای این سلول هیچ اطلاعاتی از اینترنت دریافت نمی‌کند.

In [ ]:
# تعداد روزهای تاریخچه‌ای که در همگام‌سازی دریافت می‌شود
HISTORY_DAYS = 365

book = load_book()
if not WORKBOOK_PATH.exists():
    save_book(book)

def refresh_data():
    """بارگذاری دوباره فایل Excel پس از افزودن نماد یا همگام‌سازی."""
    global book, daily_data, market_data
    book = load_book()
    daily_data = book["DailyData"].copy()
    market_data = enrich_market_data(daily_data)
    return market_data

market_data = refresh_data()
print(f"نمادهای پیگیری: {len(book['Symbols'])}")
print(f"ردیف‌های تاریخچه: {len(daily_data)}")
print(f"مسیر فایل Excel: {WORKBOOK_PATH}")

## ۳. جست‌وجو و افزودن نماد

ابتدا جست‌وجو را اجرا کن. سپس شماره‌ی ردیف نتیجه را به تابع افزودن بده. این بخش به اینترنت و دسترسی TSETMC نیاز دارد.

In [ ]:
search_results = []

def search_symbol(query: str) -> pd.DataFrame:
    """جست‌وجوی نماد در TSETMC و نمایش نتایج."""
    global search_results
    search_results = search_instruments(query)
    return pd.DataFrame(search_results)

def add_search_result(index: int, quantity: float = 0, average_cost: float = 0):
    """افزودن نتیجه‌ی انتخاب‌شده به فهرست پیگیری و ثبت مقادیر پرتفوی."""
    if not isinstance(index, int) or index < 0 or index >= len(search_results):
        raise IndexError("شماره ردیف نتیجه جست‌وجو معتبر نیست.")
    add_instrument(search_results[index], quantity=quantity, average_cost=average_cost)
    refresh_data()
    return book["Symbols"]

# نمونه:
# search_symbol("فولاد")
# add_search_result(0, quantity=0, average_cost=0)

## ۴. همگام‌سازی دستی از TSETMC

این سلول عمداً خودکار اجرا نمی‌شود. پس از افزودن نمادها، دستور اجرای همگام‌سازی را اجرا کن. اجرای آن برای هر نماد تاریخچه قیمت و داده حقیقی/حقوقی را دریافت و در Excel ذخیره می‌کند.

همگام‌سازی زمان‌بندی‌شده در پس‌زمینه نداریم؛ با بسته‌شدن Kernel هیچ کاری اجرا نخواهد شد.

In [ ]:
def run_manual_sync(days: int = HISTORY_DAYS):
    result = sync_all(trigger="NOTEBOOK_MANUAL", days=days)
    refresh_data()
    display(pd.DataFrame([{
        "وضعیت": result["status"],
        "تعداد نماد پردازش‌شده": result["symbols_processed"],
        "تعداد ردیف دریافت‌شده": result["records_upserted"],
    }]))
    if result["errors"]:
        display(pd.DataFrame({"خطاها": result["errors"]}))
    return result

# برای شروع دریافت داده، این خط را اجرا کن:
# sync_result = run_manual_sync()

## ۵. مرور داده‌های ذخیره‌شده

In [ ]:
market_data = refresh_data()

display(book["Symbols"])
display(book["Portfolio"])

if market_data.empty:
    print("هنوز تاریخچه‌ای ذخیره نشده است. ابتدا نماد اضافه و همگام‌سازی را اجرا کن.")
else:
    display(market_data.sort_values(["Symbol", "Date"]).tail(20))

## ۶. رتبه‌بندی جریان حقوقی در آخرین روز موجود

برای جلوگیری از سیگنال کاذب، رتبه‌بندی جریان حقوقی فقط روی ردیف‌هایی انجام می‌شود که داده حقیقی/حقوقی آن‌ها در منبع موجود بوده است. خالص خرید مثبت به معنی خرید خالص حقوقی و مقدار منفی به معنی فروش خالص است.

In [ ]:
if market_data.empty:
    print("داده‌ای برای رتبه‌بندی وجود ندارد.")
else:
    latest_date = market_data["Date"].max()
    latest_rows = market_data.loc[market_data["Date"].eq(latest_date)].copy()
    if "ClientDataAvailable" in latest_rows:
        valid_flow = latest_rows.loc[latest_rows["ClientDataAvailable"].fillna(False).astype(bool)]
    else:
        valid_flow = latest_rows.iloc[0:0]

    ranking_columns = [
        "Symbol", "Date", "ClosePrice", "DailyChangePct", "TotalVolume",
        "LegalBuyVolume", "LegalSellVolume", "NetLegalVolume",
        "NetLegalPctVolume", "LegalBuySharePct", "LegalSellSharePct",
    ]
    print(f"آخرین تاریخ داده: {pd.Timestamp(latest_date).date()}")
    if valid_flow.empty:
        print("برای این روز داده معتبر حقیقی/حقوقی موجود نیست.")
    else:
        display(
            valid_flow.sort_values("NetLegalVolume", ascending=False)[ranking_columns]
            .reset_index(drop=True)
        )

    availability = latest_rows.get(
        "ClientDataAvailable",
        pd.Series(False, index=latest_rows.index)
    ).fillna(False).astype(bool)
    unavailable = latest_rows.loc[~availability, ["Symbol", "Date"]]
    if not unavailable.empty:
        print("نمادهای فاقد داده حقیقی/حقوقی در آخرین روز:")
        display(unavailable.reset_index(drop=True))

## ۷. تشخیص الگوهای احتمالی انباشت و توزیع

این تشخیص بر اساس قواعد اولیه است، نه مدل پیش‌بینی و نه احتمال آماری. انباشت به خالص خرید حقوقی متوالی همراه با حرکت محدود قیمت اشاره دارد؛ توزیع به خالص فروش حقوقی نزدیک سقف اخیر. داده‌ی حقیقی/حقوقی ناموجود نباید به‌عنوان صفر تفسیر شود.

In [ ]:
signals = detect_latest_patterns(market_data)
if signals.empty:
    print("در آخرین داده‌ی نمادها الگویی مطابق قواعد فعلی شناسایی نشد.")
else:
    display(signals)

## ۸. بررسی نمودار یک نماد

In [ ]:
def plot_symbol(symbol: str, days: int = 180):
    """نمایش نمودار قیمت و جریان حقوقی برای یک نماد."""
    if market_data.empty:
        print("ابتدا داده دریافت کن.")
        return None

    history = market_data.loc[market_data["Symbol"].astype(str).eq(str(symbol))].copy()
    history = history.sort_values("Date").tail(days)
    if history.empty:
        print(f"برای نماد {symbol} داده‌ای پیدا نشد.")
        return None

    price_fig = go.Figure()
    required_ohlc = {"OpenPrice", "HighPrice", "LowPrice", "ClosePrice"}
    if required_ohlc.issubset(history.columns):
        price_fig.add_trace(go.Candlestick(
            x=history["Date"],
            open=history["OpenPrice"],
            high=history["HighPrice"],
            low=history["LowPrice"],
            close=history["ClosePrice"],
            name="قیمت",
        ))
    else:
        price_fig.add_trace(go.Scatter(
            x=history["Date"], y=history["ClosePrice"],
            mode="lines", name="قیمت پایانی",
        ))
    price_fig.update_layout(
        title=f"روند قیمت {symbol}",
        xaxis_title="تاریخ",
        yaxis_title="قیمت",
        xaxis_rangeslider_visible=False,
        height=430,
    )
    price_fig.show()

    flow_fig = go.Figure()
    flow_fig.add_trace(go.Bar(
        x=history["Date"], y=history["LegalBuyVolume"], name="خرید حقوقی"
    ))
    flow_fig.add_trace(go.Bar(
        x=history["Date"], y=-history["LegalSellVolume"], name="فروش حقوقی"
    ))
    flow_fig.add_trace(go.Scatter(
        x=history["Date"], y=history["NetLegalVolume"],
        mode="lines", name="خالص خرید حقوقی",
    ))
    flow_fig.update_layout(
        title=f"جریان خرید و فروش حقوقی {symbol}",
        xaxis_title="تاریخ",
        yaxis_title="حجم",
        barmode="relative",
        height=380,
    )
    flow_fig.show()
    return history

# نمونه:
# plot_symbol("فولاد", days=180)

## ۹. آزمون تاریخی الگوها (Backtest)

بازده‌ها با مقایسه‌ی قیمت پایانی در روز سیگنال و قیمت پایانی در جلسه‌ی معاملاتی بعد از افق انتخابی محاسبه می‌شوند. پیش از نتیجه‌گیری باید تعداد نمونه‌ها، توقف نماد، قیمت تعدیل‌نشده، افزایش سرمایه، کارمزد، نقدشوندگی و سوگیری بقا را بررسی کرد.

In [ ]:
backtest_results = backtest_patterns(market_data, horizons=HORIZONS)
if backtest_results.empty:
    print("برای آزمون تاریخی داده‌ی کافی وجود ندارد.")
else:
    display(
        backtest_results.sort_values(
            ["Type", "Horizon", "SuccessRatePct"],
            ascending=[True, True, False],
        ).reset_index(drop=True)
    )

## ۱۰. تاریخچه سیگنال‌ها و گزارش همگام‌سازی

In [ ]:
signal_history = build_signal_history(market_data)
print("آخرین رخدادهای الگو:")
display(signal_history.tail(30))

print("گزارش همگام‌سازی:")
display(book["SyncLog"].tail(30))

## نکات مهم و محدودیت‌ها

- خالص خرید حقوقی = حجم خرید حقوقی منهای حجم فروش حقوقی.
- درصد خالص خرید از حجم روز با سهم حقوقی از کل خرید یا سهم حقوقی از کل فروش یکسان نیست؛ هر سه شاخص جدا هستند.
- داده‌ی تجمیعی حقیقی/حقوقی هویت اشخاص حقوقی منفرد را نشان نمی‌دهد.
- امتیاز الگو، احتمال موفقیت نیست و هیچ سیگنالی تضمین‌کننده‌ی بازده آینده نیست.
- مسیرهای داده‌ی TSETMC در آداپتر فعلی، API رسمی با قرارداد پایداری تضمین‌شده نیستند. ساختار پاسخ، نام فیلدها یا دسترسی شبکه ممکن است تغییر کند؛ مخصوصاً دسترسی از برخی IPهای خارج ایران باید عملاً آزمایش شود.
- فایل Excel هنگام ذخیره نباید در برنامه‌ی دیگری باز باشد.
- برای پشتیبان‌گیری، از فایل data/SmartMoneyData.xlsx یک کپی جداگانه نگه دار.